
# VN History RAG Inference — Qwen2.5 + Phase1/Phase2 LoRA + FAISS

Notebook này dùng để test inference end-to-end:

1. Mount Google Drive.
2. Load `all_chunk_id.jsonl` từ dataset train phase2.
3. Build hoặc load FAISS embedding index.
4. Load Qwen2.5 vanilla → load phase1 adapter → merge → load phase2 adapter → merge optional.
5. Retrieve chunk bằng FAISS.
6. Format prompt đúng kiểu đã train SFT: chỉ có `user` và `assistant`, không có system.
7. Generate an toàn cho Qwen, stop ở `<|im_end|>`, cắt output tránh lặp `user`.
8. Test câu hỏi mẫu và sanity test bằng gold-context trong `all_messages.jsonl`.


In [1]:

# Cell 1 — Install dependencies

%pip -q install -U   "transformers>=4.45.0"   "peft>=0.13.0"   "accelerate>=0.34.0"   "sentence-transformers>=3.0.0"   "faiss-cpu>=1.8.0"   "safetensors>=0.4.0"   "tqdm>=4.66.0"


In [8]:
!pip uninstall -y torchao
!pip install -q --upgrade "torchao>=0.16.0"

# Kiểm tra version
import torchao
print("torchao version:", torchao.__version__)

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.9 MB/s eta 0:00:00


torchao version: 0.17.0


In [2]:

# Cell 2 — Mount Drive, imports, global config

from google.colab import drive
drive.mount('/content/drive')

import os, re, json, glob, time, shutil, zipfile, hashlib, random
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import torch
import faiss
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ===== Model / Adapter paths =====
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DRIVE_ROOT = "/content/drive/MyDrive"

DATA_DIR = f"{DRIVE_ROOT}/vn_history_model_backups"
CHUNKS_PATH = f"{DATA_DIR}/vn_history_rag_sft_dataset/all_chunk_id.jsonl"
MESSAGES_PATH = f"{DATA_DIR}/vn_history_rag_sft_dataset/all_messages.jsonl"

BACKUP_DIR = f"{DRIVE_ROOT}/vn_history_model_backups"

PHASE1_ADAPTER_CANDIDATES = [
    f"{BACKUP_DIR}/qwen_vnhistory_phase1_best_adapter",
    f"{BACKUP_DIR}/qwen_vnhistory_phase1_best_adapter.zip",
]

PHASE2_ADAPTER_CANDIDATES = [
    f"{BACKUP_DIR}/qwen_vnhistory_phase6_rag_best_adapter",
    f"{BACKUP_DIR}/qwen_vnhistory_phase6_rag_best_adapter.zip",
    f"{BACKUP_DIR}/qwen2_5_3b_vnhistory_phase6_rag_qlora_best_by_generation_metric",
    f"{BACKUP_DIR}/qwen2_5_3b_vnhistory_phase6_rag_qlora_best_by_generation_metric.zip",
]

# Nếu bạn đã export full merged model phase1+phase2 thì có thể set đường dẫn ở đây và bỏ qua adapter merge.
FULL_MERGED_MODEL_DIR = None

# Merge phase2 vào model sau khi load adapter để inference nhanh/gọn hơn.
# Nếu thiếu VRAM thì đổi False.
MERGE_PHASE2_FOR_INFERENCE = True

# ===== Embedding / FAISS config =====
# intfloat/multilingual-e5-base nhẹ và ổn cho tiếng Việt. Nếu muốn mạnh hơn, thử "BAAI/bge-m3".
EMBEDDING_MODEL_ID = "intfloat/multilingual-e5-base"
EMBED_BATCH_SIZE = 64
FORCE_REBUILD_FAISS = False

# FAISS search config
TOP_K = 5
FETCH_K = 30
MIN_SCORE_TO_TRUST = None  # ví dụ 0.20; để None thì luôn đưa context cho model tự quyết.

# Context budget: tránh prompt bị truncate mất <|im_start|>assistant.
MAX_MODEL_LENGTH = 4096
MAX_INPUT_TOKENS = 3600
MAX_NEW_TOKENS = 224
MAX_CHARS_PER_CHUNK = 1600
MIN_CHARS_PER_CHUNK = 500

# Generation config
TEMPERATURE = 0.0
TOP_P = 1.0
REPETITION_PENALTY = 1.05

# Cache paths
safe_embed_name = re.sub(r"[^a-zA-Z0-9._-]+", "_", EMBEDDING_MODEL_ID)
FAISS_DIR = f"{DATA_DIR}/faiss_cache_{safe_embed_name}"
FAISS_INDEX_PATH = f"{FAISS_DIR}/chunks.index"
FAISS_META_PATH = f"{FAISS_DIR}/chunks_meta.json"

print("CHUNKS_PATH:", CHUNKS_PATH)
print("MESSAGES_PATH:", MESSAGES_PATH)
print("FAISS_DIR:", FAISS_DIR)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CHUNKS_PATH: /content/drive/MyDrive/vn_history_model_backups/vn_history_rag_sft_dataset/all_chunk_id.jsonl
MESSAGES_PATH: /content/drive/MyDrive/vn_history_model_backups/vn_history_rag_sft_dataset/all_messages.jsonl
FAISS_DIR: /content/drive/MyDrive/vn_history_model_backups/faiss_cache_intfloat_multilingual-e5-base
CUDA: True NVIDIA L4


In [3]:
# Cell 3 — Utility functions

import os
import re
import json
import glob
import shutil
import zipfile
import hashlib
from typing import Any, Dict, List


def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_json(path: str, obj: Any):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s).replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


def short_text(s: str, max_chars: int) -> str:
    s = normalize_text(s)
    if len(s) <= max_chars:
        return s

    cut = s[:max_chars]

    # Cố cắt ở khoảng trắng/câu để đỡ cụt quá
    last = max(
        cut.rfind(". "),
        cut.rfind("; "),
        cut.rfind("\n"),
        cut.rfind(" "),
    )

    if last > max_chars * 0.65:
        cut = cut[:last]

    return cut.strip() + " ..."


def resolve_adapter_dir(name: str, candidates: List[str]) -> str:
    """
    Find adapter dir. If only zip exists, extract to /content/adapters/<name>.
    """
    extract_root = f"/content/adapters/{name}"
    os.makedirs("/content/adapters", exist_ok=True)

    for p in candidates:
        if not p:
            continue

        # Folder directly containing adapter_config.json
        if os.path.isdir(p) and os.path.exists(os.path.join(p, "adapter_config.json")):
            return p

        # Zip file
        if os.path.isfile(p) and p.endswith(".zip"):
            if os.path.exists(os.path.join(extract_root, "adapter_config.json")):
                return extract_root

            print(f"Extracting {name} adapter zip:", p)

            if os.path.exists(extract_root):
                shutil.rmtree(extract_root)

            os.makedirs(extract_root, exist_ok=True)

            with zipfile.ZipFile(p, "r") as z:
                z.extractall(extract_root)

            # If zip has nested folder, locate adapter_config.json
            matches = glob.glob(
                os.path.join(extract_root, "**", "adapter_config.json"),
                recursive=True,
            )

            if not matches:
                raise FileNotFoundError(f"Không thấy adapter_config.json trong zip {p}")

            return os.path.dirname(matches[0])

    # Try glob fallback for zip/folders containing name-ish
    for p in glob.glob(f"{BACKUP_DIR}/**/adapter_config.json", recursive=True):
        if name.lower() in p.lower():
            return os.path.dirname(p)

    raise FileNotFoundError(
        f"Không tìm thấy adapter {name}. Candidates:\n"
        + "\n".join(str(c) for c in candidates)
    )


def get_model_device(model):
    try:
        return model.get_input_embeddings().weight.device
    except Exception:
        return next(model.parameters()).device


def hash_file(path: str, max_bytes: int = 2_000_000) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        h.update(f.read(max_bytes))
    return h.hexdigest()[:12]

In [4]:

# Cell 4 — Load chunks from all_chunk_id.jsonl

assert os.path.exists(CHUNKS_PATH), f"Không thấy file chunks: {CHUNKS_PATH}"
raw_chunks = read_jsonl(CHUNKS_PATH)
print("Raw chunks:", len(raw_chunks))

# Deduplicate by chunk_id, giữ bản đầu tiên.
chunk_by_id = {}
chunks = []
for c in raw_chunks:
    cid = c.get("chunk_id")
    if not cid or cid in chunk_by_id:
        continue
    item = {
        "chunk_id": cid,
        "title": c.get("title", ""),
        "source_type": c.get("source_type", ""),
        "source": c.get("source", ""),
        "url": c.get("url", ""),
        "chunk_index": c.get("chunk_index", None),
        "word_len": c.get("word_len", None),
        "history_score": c.get("history_score", None),
        "text": normalize_text(c.get("text", "")),
    }
    chunk_by_id[cid] = item
    chunks.append(item)

all_chunk_ids = set(chunk_by_id.keys())
print("Dedup chunks:", len(chunks))
print("Sample:")
print(json.dumps(chunks[0], ensure_ascii=False, indent=2)[:1200])


Raw chunks: 520
Dedup chunks: 511
Sample:
{
  "chunk_id": "hf_wikipedia_ngô_quyền_0008_117505677cc6",
  "title": "Ngô Quyền",
  "source_type": "hf_wikipedia",
  "source": "DataStudio/Viet-wikipedia",
  "url": "https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E1%BB%81n",
  "chunk_index": 8,
  "word_len": 414,
  "history_score": 101,
  "text": "Tử làm thánh thành hoàng. Nhiều đường phố mang tên Ngô Quyền như tại quận Hoàn Kiếm và Hà Đông, Hà Nội, thành phố Thanh Hóa, thị xã Quảng Yên, thành phố Đà Nẵng, thành phố Quy Nhơn... Tên ông cũng là tên của một quận nội thành của Hải Phòng. Nhiều trường học ở Việt Nam cũng mang tên Ngô Quyền. Ảnh Xem thêm Khúc Thừa Dụ Dương Đình Nghệ Trận Bạch Đằng (938) Các bãi cọc trên sông Bạch Đằng Nhà Ngô Dương Tam Kha Dương Như Ngọc Ngô Xương Ngập Ngô Xương Văn Hậu Ngô Vương Chú thích Tham khảo Nhiều tác giả (1972), Đại Việt Sử ký Toàn thư, Cao Huy Giu phiên dịch, Nhà Xuất bản Khoa học Xã hội. Phan Bội Châu (1909), Việt Nam quốc sử khảo. Nhiều tác giả (1991), L

In [5]:
# Cell 5 — Build / load FAISS embedding index

os.makedirs(FAISS_DIR, exist_ok=True)

USE_E5_PREFIX = "e5" in EMBEDDING_MODEL_ID.lower()


def passage_for_embedding(c: Dict[str, Any]) -> str:
    """
    Tạo text để embedding.
    Đưa title + chunk_id lên đầu để query như "Bình Ngô đại cáo", "Lam Sơn",
    "Lý Công Uẩn" match chủ đề tốt hơn.
    """
    body = (
        f"title: {c.get('title', '')}\n"
        f"chunk_id: {c.get('chunk_id', '')}\n"
        f"text: {c.get('text', '')}"
    )

    body = normalize_text(body)

    if USE_E5_PREFIX:
        return "passage: " + body

    return body


def query_for_embedding(question: str) -> str:
    q = normalize_text(question)

    if USE_E5_PREFIX:
        return "query: " + q

    return q


def load_embedding_model():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Loading embedding model:", EMBEDDING_MODEL_ID, "device=", device)

    embedder = SentenceTransformer(
        EMBEDDING_MODEL_ID,
        device=device,
    )

    return embedder


need_rebuild = FORCE_REBUILD_FAISS or not (
    os.path.exists(FAISS_INDEX_PATH)
    and os.path.exists(FAISS_META_PATH)
)

embedder = load_embedding_model()

if not need_rebuild:
    print("Loading FAISS index from cache:", FAISS_INDEX_PATH)

    index = faiss.read_index(FAISS_INDEX_PATH)

    with open(FAISS_META_PATH, "r", encoding="utf-8") as f:
        faiss_meta = json.load(f)

    print("Loaded index ntotal:", index.ntotal, "meta:", len(faiss_meta))

else:
    print("Building FAISS index from chunks...")

    # Dedup theo chunk_id để tránh retrieve trùng chunk.
    dedup_chunks = []
    seen_chunk_ids = set()

    for c in chunks:
        cid = c.get("chunk_id")

        if not cid:
            continue

        if cid in seen_chunk_ids:
            continue

        seen_chunk_ids.add(cid)
        dedup_chunks.append(c)

    print("Original chunks:", len(chunks))
    print("Dedup chunks:", len(dedup_chunks))

    docs = [passage_for_embedding(c) for c in dedup_chunks]

    embeddings = embedder.encode(
        docs,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    dim = embeddings.shape[1]

    # Cosine similarity vì embeddings đã normalize.
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    faiss_meta = dedup_chunks

    faiss.write_index(index, FAISS_INDEX_PATH)
    write_json(FAISS_META_PATH, faiss_meta)

    print("Saved FAISS index:", FAISS_INDEX_PATH)
    print("Saved FAISS meta:", FAISS_META_PATH)
    print("Index ntotal:", index.ntotal, "dim:", dim)

assert index.ntotal == len(faiss_meta), (index.ntotal, len(faiss_meta))

Loading embedding model: intfloat/multilingual-e5-base device= cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading FAISS index from cache: /content/drive/MyDrive/vn_history_model_backups/faiss_cache_intfloat_multilingual-e5-base/chunks.index
Loaded index ntotal: 511 meta: 511


In [6]:
# Cell 6 — Retriever with FAISS + dedup + context budget

from typing import Any, Dict, List, Tuple


def retrieve_chunks(
    question: str,
    top_k: int = TOP_K,
    fetch_k: int = FETCH_K,
) -> List[Dict[str, Any]]:
    q = query_for_embedding(question)

    q_emb = embedder.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    scores, idxs = index.search(q_emb, min(fetch_k, index.ntotal))

    results = []
    seen = set()

    for score, idx in zip(scores[0].tolist(), idxs[0].tolist()):
        if idx < 0:
            continue

        c = dict(faiss_meta[idx])
        cid = c.get("chunk_id")

        if not cid:
            continue

        if cid in seen:
            continue

        seen.add(cid)
        c["score"] = float(score)
        results.append(c)

        if len(results) >= top_k:
            break

    return results


def build_context_text(
    chunks_for_context: List[Dict[str, Any]],
    max_chars_per_chunk: int = MAX_CHARS_PER_CHUNK,
) -> str:
    parts = []

    for c in chunks_for_context:
        cid = c.get("chunk_id", "")
        title = c.get("title", "")
        text = short_text(c.get("text", ""), max_chars_per_chunk)

        parts.append(
            f"[{cid}] {title}\n"
            f"{text}"
        )

    return "\n\n".join(parts).strip()


IM_START = "<|im_start|>"
IM_END = "<|im_end|>"


def build_user_text(
    question: str,
    chunks_for_context: List[Dict[str, Any]],
    max_chars_per_chunk: int = MAX_CHARS_PER_CHUNK,
) -> str:
    context_text = build_context_text(
        chunks_for_context,
        max_chars_per_chunk=max_chars_per_chunk,
    )

    return (
        "Câu hỏi:\n"
        f"{normalize_text(question)}\n\n"
        "Tài liệu tham khảo:\n"
        f"{context_text}"
    ).strip()


def build_prompt_from_user_text(user_text: str) -> str:
    """
    Đúng style SFT đã train: chỉ user + assistant, không thêm system.
    Quan trọng: prompt phải kết thúc bằng <|im_start|>assistant\\n
    để model hiểu đây là lúc cần trả lời, không phải tiếp tục viết context.
    """
    return (
        f"{IM_START}user\n"
        f"{user_text}"
        f"{IM_END}\n"
        f"{IM_START}assistant\n"
    )


def token_len(text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def make_fitted_prompt(
    question: str,
    retrieved: List[Dict[str, Any]],
    max_input_tokens: int = MAX_INPUT_TOKENS,
) -> Tuple[str, List[Dict[str, Any]], Dict[str, Any]]:
    """
    Fit prompt vào token budget bằng cách:
    1. giảm số ký tự mỗi chunk;
    2. nếu vẫn quá dài thì giảm số chunk.

    Mục tiêu chính:
    - KHÔNG dùng tokenizer truncation để cắt prompt.
    - KHÔNG để mất đoạn cuối <|im_start|>assistant.
    """
    used = list(retrieved)
    chars = MAX_CHARS_PER_CHUNK

    while used:
        user_text = build_user_text(
            question,
            used,
            max_chars_per_chunk=chars,
        )

        prompt = build_prompt_from_user_text(user_text)
        n_tok = token_len(prompt)

        if n_tok <= max_input_tokens:
            info = {
                "input_tokens": n_tok,
                "chars_per_chunk": chars,
                "n_context_chunks": len(used),
                "used_chunk_ids": [c.get("chunk_id", "") for c in used],
            }
            return prompt, used, info

        if chars > MIN_CHARS_PER_CHUNK:
            chars = max(MIN_CHARS_PER_CHUNK, int(chars * 0.75))
        else:
            used = used[:-1]

    # Fallback: không có chunk nào, vẫn hỏi model với context rỗng.
    user_text = (
        "Câu hỏi:\n"
        f"{normalize_text(question)}\n\n"
        "Tài liệu tham khảo:\n"
    )

    prompt = build_prompt_from_user_text(user_text)

    info = {
        "input_tokens": token_len(prompt),
        "chars_per_chunk": 0,
        "n_context_chunks": 0,
        "used_chunk_ids": [],
    }

    return prompt, [], info


def debug_prompt_tail(prompt: str, n_chars: int = 500):
    """
    Dùng để kiểm tra prompt có bị mất assistant header không.
    Nếu cuối prompt không có <|im_start|>assistant thì inference rất dễ loạn.
    """
    print("=" * 100)
    print("PROMPT TOKEN LEN:", token_len(prompt))
    print("PROMPT TAIL:")
    print(prompt[-n_chars:])
    print("=" * 100)

    if not prompt.endswith(f"{IM_START}assistant\n"):
        print("WARNING: Prompt không kết thúc bằng assistant header.")
    else:
        print("OK: Prompt kết thúc đúng bằng assistant header.")


# Quick retrieval smoke test
# Quick retrieval smoke test
q = "Bình Ngô đại cáo ra đời trong bối cảnh nào?"
retr = retrieve_chunks(q, top_k=5)

print("Query:", q)
print("Retrieved chunks:")

for r in retr:
    print(f"{r['score']:.4f} | {r['chunk_id']} | {r.get('title', '')}")

# Chỉ test fit prompt nếu tokenizer đã được load.
# Nếu chưa load tokenizer thì bỏ qua, vì make_fitted_prompt cần token_len().
if "tokenizer" in globals():
    prompt, used_chunks, fit_info = make_fitted_prompt(q, retr)

    print("\nFit info:")
    print(json.dumps(fit_info, ensure_ascii=False, indent=2))

    debug_prompt_tail(prompt)
else:
    print("\nTokenizer chưa được load, nên bỏ qua make_fitted_prompt/debug_prompt_tail ở Cell 6.")
    print("Sau khi chạy cell load model/tokenizer, bạn có thể test lại:")
    print("prompt, used_chunks, fit_info = make_fitted_prompt(q, retr)")
    print("debug_prompt_tail(prompt)")

Query: Bình Ngô đại cáo ra đời trong bối cảnh nào?
Retrieved chunks:
0.8403 | hf_wikipedia_ngô_quyền_0000_af223d790816 | Ngô Quyền
0.8188 | hf_wikipedia_ngô_họ_0000_82a61434b158 | Ngô (họ)
0.8102 | hf_wikipedia_ngô_quyền_0002_91f9c47eaa2a | Ngô Quyền
0.8073 | hf_wikipedia_ngô_quyền_0001_c00ce6aaa0d9 | Ngô Quyền
0.8053 | hf_wikipedia_nhà_lê_sơ_0012_7d87415a8d27 | Nhà Lê sơ

Tokenizer chưa được load, nên bỏ qua make_fitted_prompt/debug_prompt_tail ở Cell 6.
Sau khi chạy cell load model/tokenizer, bạn có thể test lại:
prompt, used_chunks, fit_info = make_fitted_prompt(q, retr)
debug_prompt_tail(prompt)


In [9]:

# Cell 7 — Load Qwen2.5 + phase1 adapter merge + phase2 adapter merge

if FULL_MERGED_MODEL_DIR and os.path.exists(FULL_MERGED_MODEL_DIR):
    print("Loading full merged model:", FULL_MERGED_MODEL_DIR)
    tokenizer = AutoTokenizer.from_pretrained(FULL_MERGED_MODEL_DIR, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        FULL_MERGED_MODEL_DIR,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    PHASE1_ADAPTER_DIR = resolve_adapter_dir("phase1", PHASE1_ADAPTER_CANDIDATES)
    PHASE2_ADAPTER_DIR = resolve_adapter_dir("phase2", PHASE2_ADAPTER_CANDIDATES)

    print("PHASE1_ADAPTER_DIR:", PHASE1_ADAPTER_DIR)
    print("PHASE2_ADAPTER_DIR:", PHASE2_ADAPTER_DIR)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    print("Loading vanilla base:", MODEL_ID)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )

    base_model.config.pad_token_id = tokenizer.pad_token_id

    print("Loading phase1 adapter onto vanilla base...")
    model = PeftModel.from_pretrained(base_model, PHASE1_ADAPTER_DIR, is_trainable=False)

    print("Merging phase1 adapter into base...")
    model = model.merge_and_unload()
    torch.cuda.empty_cache()

    print("Loading phase2 RAG adapter onto phase1-merged base...")
    model = PeftModel.from_pretrained(model, PHASE2_ADAPTER_DIR, is_trainable=False)

    if MERGE_PHASE2_FOR_INFERENCE:
        print("Merging phase2 adapter for inference...")
        model = model.merge_and_unload()
        torch.cuda.empty_cache()

model.eval()
model.config.use_cache = True
model.config.pad_token_id = tokenizer.pad_token_id

print("Model ready.")
print("Device:", get_model_device(model))
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token_id:", tokenizer.eos_token_id)
print("im_end_id:", tokenizer.convert_tokens_to_ids(IM_END))


PHASE1_ADAPTER_DIR: /content/drive/MyDrive/vn_history_model_backups/qwen_vnhistory_phase1_best_adapter
PHASE2_ADAPTER_DIR: /content/drive/MyDrive/vn_history_model_backups/qwen_vnhistory_phase6_rag_best_adapter
Loading vanilla base: Qwen/Qwen2.5-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading phase1 adapter onto vanilla base...
Merging phase1 adapter into base...
Loading phase2 RAG adapter onto phase1-merged base...
Merging phase2 adapter for inference...
Model ready.
Device: cuda:0
pad_token_id: 151643
eos_token_id: 151645
im_end_id: 151645


In [10]:
# Cell 8 — Safe Qwen generation + source parsing

import re
import torch
from typing import Any, Dict, List


ANSWER_SPLIT_RE = re.compile(r"Trả lời\s*:", re.IGNORECASE)

SOURCE_LINE_RE = re.compile(
    r"Nguồn được dùng\s*:\s*(.*?)(?:\n\s*\n|\n\s*Trả lời\s*:|$)",
    re.IGNORECASE | re.DOTALL,
)

BRACKET_ID_RE = re.compile(r"\[([^\[\]]*)\]")


def clean_qwen_generated_text(text: str) -> str:
    if text is None:
        return ""

    # Cắt tại các marker hội thoại nếu model sinh tiếp lượt mới.
    cut_markers = [
        IM_END,
        f"{IM_START}user",
        f"{IM_START}system",
        f"{IM_START}assistant",
        "\nuser\n",
        "\nUser\n",
        "\nUSER\n",
        "\nassistant\n",
        "\nAssistant\n",
    ]

    for marker in cut_markers:
        if marker and marker in text:
            text = text.split(marker)[0]

    # Xóa token đặc biệt còn sót.
    special_tokens = [
        IM_START,
        IM_END,
        tokenizer.eos_token or "",
        tokenizer.pad_token or "",
    ]

    for tok in special_tokens:
        if tok:
            text = text.replace(tok, "")

    return normalize_text(text)


def extract_source_ids(answer: str) -> List[str]:
    m = SOURCE_LINE_RE.search(answer or "")

    if not m:
        return []

    line = m.group(1).strip()

    if line in ["[]", "", "[ ]"]:
        return []

    ids = []

    # Parse cả dạng [id1, id2] và [id1], [id2]
    for inside in BRACKET_ID_RE.findall(line):
        for part in inside.split(","):
            part = part.strip().strip("'\"")

            if part and part not in ids:
                ids.append(part)

    return ids


def extract_answer_body(answer: str) -> str:
    parts = ANSWER_SPLIT_RE.split(answer or "", maxsplit=1)

    if len(parts) > 1:
        return normalize_text(parts[1])

    return normalize_text(answer)


@torch.inference_mode()
def generate_from_prompt(
    prompt: str,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> str:
    device = get_model_device(model)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = True
    model.eval()

    # Không dùng truncation ở đây; prompt đã được fit trước đó.
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=False,
        truncation=False,
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]

    im_end_id = tokenizer.convert_tokens_to_ids(IM_END)

    eos_ids = []

    if isinstance(tokenizer.eos_token_id, int) and tokenizer.eos_token_id >= 0:
        eos_ids.append(tokenizer.eos_token_id)

    if isinstance(im_end_id, int) and im_end_id >= 0 and im_end_id not in eos_ids:
        eos_ids.append(im_end_id)

    if not eos_ids:
        eos_ids = None

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=TEMPERATURE > 0,
        eos_token_id=eos_ids,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
        repetition_penalty=REPETITION_PENALTY,
    )

    if TEMPERATURE > 0:
        gen_kwargs["temperature"] = TEMPERATURE
        gen_kwargs["top_p"] = TOP_P

    out = model.generate(**gen_kwargs)

    new_tokens = out[0][prompt_len:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=False)

    return clean_qwen_generated_text(raw)


def answer_question(
    question: str,
    top_k: int = TOP_K,
    fetch_k: int = FETCH_K,
    verbose: bool = True,
) -> Dict[str, Any]:
    retrieved = retrieve_chunks(
        question,
        top_k=top_k,
        fetch_k=fetch_k,
    )

    # Nếu top score quá thấp thì có thể đưa context rỗng để model trả insufficient.
    if MIN_SCORE_TO_TRUST is not None and retrieved and retrieved[0]["score"] < MIN_SCORE_TO_TRUST:
        retrieved_for_prompt = []
    else:
        retrieved_for_prompt = retrieved

    prompt, fitted_chunks, prompt_info = make_fitted_prompt(
        question,
        retrieved_for_prompt,
    )

    # Safety: xác nhận cuối prompt vẫn có assistant header.
    assert prompt.endswith(f"{IM_START}assistant\n"), "Prompt bị mất assistant header"
    assert token_len(prompt) <= MAX_INPUT_TOKENS, (token_len(prompt), MAX_INPUT_TOKENS)

    answer = generate_from_prompt(prompt)
    used_ids = extract_source_ids(answer)

    result = {
        "question": question,
        "retrieved": [
            {
                "chunk_id": c["chunk_id"],
                "title": c.get("title", ""),
                "score": c.get("score", None),
            }
            for c in retrieved
        ],
        "context_chunk_ids": [c["chunk_id"] for c in fitted_chunks],
        "prompt_info": prompt_info,
        "used_chunk_ids": used_ids,
        "answer": answer,
        "answer_body": extract_answer_body(answer),
    }

    if verbose:
        print("=" * 110)
        print("QUESTION:", question)

        print("\nRETRIEVED CHUNKS:")
        for c in retrieved:
            print(
                f"- {c['chunk_id']} | {c.get('title', '')} | "
                f"score={c.get('score', 0):.4f}"
            )

        print("\nCONTEXT USED:", result["context_chunk_ids"])
        print("PROMPT INFO:", prompt_info)
        print("USED CHUNK IDS:", used_ids)

        print("\nANSWER:")
        print(answer)

    return result

In [12]:

# Cell 9 — Smoke test: check prompt fitting does not truncate assistant header

question = "Việc Lý Công Uẩn dời đô ra Thăng Long năm 1010 có ý nghĩa gì?"
retrieved = retrieve_chunks(question, top_k=TOP_K, fetch_k=FETCH_K)
prompt, fitted_chunks, info = make_fitted_prompt(question, retrieved)

print("Prompt info:", info)
print("Prompt token len:", token_len(prompt))
print("Prompt tail:")
print(tokenizer.decode(tokenizer(prompt, add_special_tokens=False)["input_ids"][-120:], skip_special_tokens=False))
print("Fitted chunk ids:", [c["chunk_id"] for c in fitted_chunks])


Prompt info: {'input_tokens': 2666, 'chars_per_chunk': 1600, 'n_context_chunks': 5, 'used_chunk_ids': ['hf_wikipedia_nhà_lý_0002_830b4c6c4cba', 'hf_wikipedia_nhà_lý_0003_43d1ecf9570b', 'hf_wikipedia_đại_cồ_việt_0004_52bff329f1ec', 'hf_wikipedia_quần_thể_di_tích_cố_đô_hoa_lư_0001_600b5db3ec39', 'hf_wikipedia_ngô_quyền_0003_5da5ba8598ed']}
Prompt token len: 2666
Prompt tail:
 dấu vết cả về tự nhiên và xã hội của phương Bắc, và thế lực của họ ở Đại La không phải nhỏ. Do đó lực lượng này dễ thực hiện việc tiếp tay làm nội ứng khi quân phương Bắc trở lại, điển hình là việc Khúc Thừa Mỹ nhanh chóng thất bại và bị Nam Hán bắt về Phiên Ngung. Rút kinh nghiệm từ thất bại của Khúc Thừa Mỹ, Ngô Quyền không chọn Đại La. Theo Tạ Chí Đại Trường: Chiếm giữ Đại La xong, Ngô ...<|im_end|>
<|im_start|>assistant

Fitted chunk ids: ['hf_wikipedia_nhà_lý_0002_830b4c6c4cba', 'hf_wikipedia_nhà_lý_0003_43d1ecf9570b', 'hf_wikipedia_đại_cồ_việt_0004_52bff329f1ec', 'hf_wikipedia_quần_thể_di_tích_cố_đô_hoa_lư_000

In [13]:

# Cell 10 — Manual test questions

test_questions = [
    "Chiến thắng Bạch Đằng năm 938 có ý nghĩa lịch sử gì?",
    "Việc Lý Công Uẩn dời đô ra Thăng Long năm 1010 có ý nghĩa gì?",
    "Bình Ngô đại cáo ra đời trong bối cảnh nào?",
    "Khởi nghĩa Lam Sơn diễn ra trong bối cảnh nào và kết quả ra sao?",
    "So sánh vai trò của nhà Lý và nhà Trần trong xây dựng và bảo vệ Đại Việt.",
    "Phong trào Cần Vương bùng nổ trong hoàn cảnh nào?",
    "Xô Viết Nghệ Tĩnh 1930-1931 có đặc điểm gì nổi bật?",
    "Cách mạng tháng Tám năm 1945 diễn ra như thế nào và kết quả ra sao?",
    "Hiệp định Genève năm 1954 có nội dung/kết quả gì đối với Việt Nam?",
]

results = []
for q in test_questions:
    results.append(answer_question(q, top_k=TOP_K, fetch_k=FETCH_K, verbose=True))


QUESTION: Chiến thắng Bạch Đằng năm 938 có ý nghĩa lịch sử gì?

RETRIEVED CHUNKS:
- hf_wikipedia_trận_bạch_đằng_1288_0000_9e0fa2aca313 | Trận Bạch Đằng (1288) | score=0.8484
- hf_wikipedia_trận_bạch_đằng_1288_0006_637c211804e9 | Trận Bạch Đằng (1288) | score=0.8395
- hf_wikipedia_trận_bạch_đằng_1288_0001_11581a43ec9d | Trận Bạch Đằng (1288) | score=0.8205
- hf_wikipedia_ngô_quyền_0001_c00ce6aaa0d9 | Ngô Quyền | score=0.8185
- hf_wikipedia_âu_lạc_0002_526ba817a8e0 | Âu Lạc | score=0.8148

CONTEXT USED: ['hf_wikipedia_trận_bạch_đằng_1288_0000_9e0fa2aca313', 'hf_wikipedia_trận_bạch_đằng_1288_0006_637c211804e9', 'hf_wikipedia_trận_bạch_đằng_1288_0001_11581a43ec9d', 'hf_wikipedia_ngô_quyền_0001_c00ce6aaa0d9', 'hf_wikipedia_âu_lạc_0002_526ba817a8e0']
PROMPT INFO: {'input_tokens': 2582, 'chars_per_chunk': 1600, 'n_context_chunks': 5, 'used_chunk_ids': ['hf_wikipedia_trận_bạch_đằng_1288_0000_9e0fa2aca313', 'hf_wikipedia_trận_bạch_đằng_1288_0006_637c211804e9', 'hf_wikipedia_trận_bạch_đằng_1288_

In [15]:
# Cell 11 — Gold-context sanity test from all_messages.jsonl
# Mục tiêu:
# Nếu gold-context trả tốt nhưng RAG manual trả tệ thì lỗi nằm ở retriever/prompt budget,
# không phải train/model.

if os.path.exists(MESSAGES_PATH):
    messages = read_jsonl(MESSAGES_PATH)
    print("Loaded messages:", len(messages))

    def get_user_and_gold(sample):
        user_text, gold_text = "", ""

        for m in sample.get("messages", []):
            if m.get("role") == "user":
                user_text = m.get("content", "")
            elif m.get("role") == "assistant":
                gold_text = m.get("content", "")

        return user_text, gold_text


    @torch.inference_mode()
    def answer_from_gold_user_text(user_text: str) -> str:
        prompt = build_prompt_from_user_text(user_text)
        n = token_len(prompt)

        print("gold-context prompt tokens:", n)

        if n > MAX_INPUT_TOKENS:
            print(
                "WARNING: gold prompt quá dài; notebook sẽ không truncate "
                "để tránh mất assistant header."
            )

        return generate_from_prompt(prompt)


    def extract_question_from_user_text(user_text: str) -> str:
        """
        Parse câu hỏi từ user_text dạng:
        Câu hỏi:
        ...

        Tài liệu tham khảo:
        ...
        """
        m = re.search(
            r"Câu hỏi:\s*(.*?)\n\s*\n\s*Tài liệu tham khảo:",
            user_text,
            flags=re.DOTALL | re.IGNORECASE,
        )

        if m:
            return normalize_text(m.group(1))

        # fallback nếu format hơi khác
        m = re.search(
            r"Câu hỏi:\s*(.*)",
            user_text,
            flags=re.DOTALL | re.IGNORECASE,
        )

        if m:
            text = m.group(1)
            text = text.split("Tài liệu tham khảo:")[0]
            return normalize_text(text)

        return ""


    # Lấy vài sample đa dạng type
    chosen = []
    seen_types = set()

    for s in messages:
        t = s.get("type", "")

        if t not in seen_types:
            chosen.append(s)
            seen_types.add(t)

        if len(chosen) >= 6:
            break


    for s in chosen:
        user_text, gold = get_user_and_gold(s)
        question = extract_question_from_user_text(user_text)

        print("=" * 110)
        print("ID:", s.get("id"), "TYPE:", s.get("type"))
        print("QUESTION:", question)

        pred = answer_from_gold_user_text(user_text)

        print("\nPRED:")
        print(pred)

        print("\nGOLD:")
        print(gold)

else:
    print("Không thấy MESSAGES_PATH:", MESSAGES_PATH)

Loaded messages: 1000
ID: sample_0001 TYPE: noisy_context
QUESTION: Chiến thắng Bạch Đằng năm 938 gắn với nhân vật nào?
gold-context prompt tokens: 2179

PRED:
Nguồn được dùng: [hf_wikipedia_ngô_quyền_0000_af223d790816]

Trả lời:
Theo tài liệu, chiến thắng Bạch Đằng năm 938 gắn với Ngô Quyền.

GOLD:
Nguồn được dùng: [hf_wikipedia_ngô_quyền_0000_af223d790816]

Trả lời:
Chiến thắng Bạch Đằng năm 938 gắn với Ngô Quyền. Tài liệu nêu rằng năm 938, ông lãnh đạo nhân dân đánh bại quân Nam Hán trong trận Bạch Đằng.
ID: sample_0014 TYPE: grounded_qa
QUESTION: Ngô Quyền sinh và mất vào thời gian nào theo tài liệu?
gold-context prompt tokens: 795

PRED:
Nguồn được dùng: [hf_wikipedia_ngô_quyền_0000_af223d790816]

Trả lời:
Theo tài liệu, Ngô Quyền sinh ngày 17 tháng 4 năm 898 và mất ngày 14 tháng 2 năm 944.

GOLD:
Nguồn được dùng: [hf_wikipedia_ngô_quyền_0000_af223d790816]

Trả lời:
Theo tài liệu, Ngô Quyền sinh ngày 17 tháng 4 năm 898 và mất ngày 14 tháng 2 năm 944.
ID: sample_0018 TYPE: insuffic

In [ ]:

# Cell 12 — Optional: save manual test results

SAVE_RESULTS = False
OUT_PATH = f"{DATA_DIR}/inference_test_results_faiss_qwen_phase1_phase2.json"

if SAVE_RESULTS:
    write_json(OUT_PATH, results)
    print("Saved:", OUT_PATH)
else:
    print("SAVE_RESULTS=False, không lưu file.")
